In [2]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# code flow
* load the dataset
* basic preprocessing
* training process
   * create the model
   * forward pass
   * loss calculate
   * parameters update
* model evaluate

In [3]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [4]:
df.shape

(569, 31)

In [5]:
df.columns

Index(['mean radius', 'mean texture', 'mean perimeter', 'mean area',
       'mean smoothness', 'mean compactness', 'mean concavity',
       'mean concave points', 'mean symmetry', 'mean fractal dimension',
       'radius error', 'texture error', 'perimeter error', 'area error',
       'smoothness error', 'compactness error', 'concavity error',
       'concave points error', 'symmetry error', 'fractal dimension error',
       'worst radius', 'worst texture', 'worst perimeter', 'worst area',
       'worst smoothness', 'worst compactness', 'worst concavity',
       'worst concave points', 'worst symmetry', 'worst fractal dimension',
       'target'],
      dtype='object')

In [6]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, :-1], df.iloc[:, -1], test_size=0.2)

In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#Numpy arrays to pytorch tensors

In [8]:
X_train_tensor = torch.from_numpy(X_train_scaled)
X_test_tensor = torch.from_numpy(X_test_scaled)
y_train_tensor = torch.from_numpy(y_train.values)
y_test_tensor = torch.from_numpy(y_test.values)

# Define the model

In [20]:
class MySimpleNN():
  def __init__(self, X):
    self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64, requires_grad=True)
    self.bias = torch.zeros(1, dtype=torch.float64, requires_grad=True)

  def forward(self, X):
    z = torch.matmul(X, self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_function(self, y_pred, y):
    # clamp predictions to avoid log(0)
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    loss = -(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred)).mean()
    return loss


In [24]:
lr = 0.1
epochs = 25

# Training Pipeline

In [27]:
# create model
model = MySimpleNN(X_train_tensor)

for epoch in range(epochs):
  # forward pass
  y_pred = model.forward(X_train_tensor)
  # print(y_pred)

  # loss calculate
  loss = model.loss_function(y_pred, y_train_tensor)

  # backward pass
  loss.backward()

  # parameters update
  with torch.no_grad():
    model.weights -= lr * model.weights.grad
    model.bias -= lr * model.bias.grad

  # zero gradients
  model.weights.grad.zero_()
  model.bias.grad.zero_()

  # print loss for each epoch
  print(f'Epoch: {epoch + 1}, loss: {loss.item()}')




Epoch: 1, loss: 3.8652239245090882
Epoch: 2, loss: 3.7029111212965726
Epoch: 3, loss: 3.5357531696640674
Epoch: 4, loss: 3.364366987526526
Epoch: 5, loss: 3.191418100880906
Epoch: 6, loss: 3.015065743950678
Epoch: 7, loss: 2.836636174330654
Epoch: 8, loss: 2.6567869560919144
Epoch: 9, loss: 2.4764187010727525
Epoch: 10, loss: 2.295147373906197
Epoch: 11, loss: 2.1150637608943197
Epoch: 12, loss: 1.9382190269104278
Epoch: 13, loss: 1.7695129208027114
Epoch: 14, loss: 1.610642524269748
Epoch: 15, loss: 1.4634277236879227
Epoch: 16, loss: 1.3292660183759655
Epoch: 17, loss: 1.2066836735039874
Epoch: 18, loss: 1.098683059156482
Epoch: 19, loss: 1.007272489685365
Epoch: 20, loss: 0.9334273930816639
Epoch: 21, loss: 0.8760738341990278
Epoch: 22, loss: 0.833203775264678
Epoch: 23, loss: 0.8021512464307534
Epoch: 24, loss: 0.7800563482089338
Epoch: 25, loss: 0.7643205704285851


# Evaluation

In [38]:
# model evaluation
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.5).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')


Accuracy: 0.548014760017395
